In [ ]:
# CELL 1 — FIRST thing, before any imports
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # must be before torch import
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print(f"GPUs visible: {torch.cuda.device_count()}")  # must print 1

In [ ]:
# CELL 2 — Imports & config
import math
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
DEVICE     = "cuda:0"
LORA_R     = 437
LORA_ALPHA = 874
BLOCK_SIZE = 512

In [ ]:
# CELL 3 — Load model + apply LoRA
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map=DEVICE,
)

lora_config = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    target_modules = ["q_proj", "k_proj", "v_proj"],
    lora_dropout   = 0.0,
    bias           = "none",
    task_type      = TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable/1e6:.2f}M")

# Verify single GPU
free, total = torch.cuda.mem_get_info(0)
print(f"GPU 0: used={(total-free)/1e9:.2f}GB  free={free/1e9:.2f}GB")

In [ ]:
# CELL 4 — WikiText-2
raw_datasets = load_dataset("wikitext", "wikitext-2-raw-v1")

tokenized = raw_datasets.map(
    lambda ex: tokenizer(ex["text"]),
    batched=True,
    remove_columns=["text"],
    desc="Tokenising",
)

def group_texts(examples):
    concat = {k: sum(examples[k], []) for k in examples.keys()}
    total  = (len(concat["input_ids"]) // BLOCK_SIZE) * BLOCK_SIZE
    result = {k: [v[i:i+BLOCK_SIZE] for i in range(0, total, BLOCK_SIZE)]
              for k, v in concat.items()}
    result["labels"] = result["input_ids"].copy()
    return result

lm_datasets = tokenized.map(group_texts, batched=True, desc="Grouping")
print(f"Train={len(lm_datasets['train'])}  Val={len(lm_datasets['validation'])}  Test={len(lm_datasets['test'])}")

In [ ]:
import numpy as np

def compute_metrics(eval_preds):
    logits, labels = eval_preds

    # Shift for next-token prediction
    preds = np.argmax(logits[:, :-1, :], axis=-1)
    labels = labels[:, 1:]

    # Flatten
    preds = preds.reshape(-1)
    labels = labels.reshape(-1)

    # Ignore padding tokens (-100)
    mask = labels != -100

    preds = preds[mask]
    labels = labels[mask]

    accuracy = (preds == labels).mean()

    return {
        "accuracy": accuracy
    }

In [ ]:
# CELL 5 — Fine-tune
OUTPUT_DIR = "./lora_tinyllama_wikitext"

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 1,
    per_device_eval_batch_size  = 1,
    gradient_accumulation_steps = 32,
    learning_rate               = 2e-4,
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    warmup_steps                = 100,
    optim                       = "adamw_torch",
    fp16                        = True,
    logging_steps               = 50,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",
    dataloader_num_workers      = 2,
    use_cpu                     = False,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = lm_datasets["train"],
    eval_dataset  = lm_datasets["validation"],
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("Fine-tuning LoRA …")
trainer.train()
print("Done.")

In [ ]:
# CELL 10 — Test perplexity

results    = trainer.evaluate(eval_dataset=lm_datasets["test"])
perplexity = math.exp(results["eval_loss"])

print("=" * 50)
print(f"Test loss  : {results['eval_loss']:.4f}")
print(f"Perplexity : {perplexity:.2f}")
print(f"Params     : {trainable/1e6:.2f}M")
print("=" * 50)

In [ ]:
# CELL 10 — Test perplexity + accuracy

import math

results    = trainer.evaluate(eval_dataset=lm_datasets["test"])
perplexity = math.exp(results["eval_loss"])

# Next token prediction accuracy
model.eval()
correct = 0
total   = 0

with torch.no_grad():
    for batch in torch.utils.data.DataLoader(
        lm_datasets["test"],
        batch_size=1,
        collate_fn=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    ):
        input_ids = batch["input_ids"].to("cuda:0")
        labels    = batch["labels"].to("cuda:0")

        outputs      = model(input_ids=input_ids)
        shift_logits = outputs.logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        preds   = shift_logits.argmax(dim=-1)
        mask    = shift_labels != -100
        correct += (preds[mask] == shift_labels[mask]).sum().item()
        total   += mask.sum().item()

accuracy = 100.0 * correct / total

print("=" * 50)
print(f"Test loss  : {results['eval_loss']:.4f}")
print(f"Perplexity : {perplexity:.2f}")
print(f"Accuracy   : {accuracy:.2f}%")
print(f"Params     : {trainable/1e6:.2f}M  (rank={RANK}, level={LEVEL})")
print("=" * 50)